In [3]:
import numpy as np 

In [40]:
e_size = 10
h_size = 15
rng = np.random.default_rng(42)
embedding_vector = rng.random(10)
hidden_vector = rng.random(15)
W_h = rng.random((15,15))
W_e = rng.random((15, 10))
b = 1

In [41]:
new_hidden_vector = np.tanh(W_h @ hidden_vector + W_e @ embedding_vector + b) 

In [43]:
new_hidden_vector

array([0.99999969, 0.99999949, 0.99999882, 0.99999501, 0.99999998,
       0.99999994, 0.99999968, 1.        , 0.99999999, 0.99999996,
       0.99999999, 0.9999996 , 0.99999987, 1.        , 0.99999995])

In [16]:
new_hidden_vector.shape

(15,)

In [44]:
rng = np.random.default_rng(42)
sample_size = 4    
embedding_vectors = rng.random((sample_size, e_size))
hidden_vectors = []
h_0 = rng.random(h_size)
hidden_vectors.append(h_0)
b = 1
# hidden_vectors = [h_0], embedding_vectors = [e_1, e_2, e_3, e_4]

W_h = rng.random((h_size, h_size))
W_e = rng.random((h_size, e_size))
for i in range(sample_size):
    cur_hidden_vector = hidden_vectors[i]
    cur_embedding_vector = embedding_vectors[i]
    new_hidden_vector = np.tanh(W_h @ cur_hidden_vector + W_e @ cur_embedding_vector + b)
    hidden_vectors.append(new_hidden_vector)
print(hidden_vectors)

[array([0.43715192, 0.8326782 , 0.7002651 , 0.31236664, 0.8322598 ,
       0.80476436, 0.38747838, 0.2883281 , 0.6824955 , 0.13975248,
       0.1999082 , 0.00736227, 0.78692438, 0.66485086, 0.70516538]), array([0.9999828 , 0.99999988, 0.99999868, 0.99999954, 0.99999991,
       0.99999997, 0.99999993, 0.99999984, 0.99999989, 0.99999432,
       0.99999998, 0.99999996, 0.99999993, 0.9999995 , 0.99999423]), array([0.99999999, 1.        , 0.99999998, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 0.99999963]), array([0.99999998, 1.        , 0.99999999, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 0.99999983]), array([0.99999998, 1.        , 0.99999998, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        

In [45]:
def softmax(x):
    max_value = np.max(x)
    x = x - max_value
    x =  np.exp(x)
    return x / sum(x)

In [ ]:
def rnn_step_forward(e_t, h_prev, parameters):
    # Compute y_hat_t, h_t
    W_e , W_h , b1, W_y, b2 = parameters["W_e"], parameters["W_h"], parameters["b1"], parameters["W_y"], parameters["b2"]
    tmp = W_e @ e_t + W_h @ h_prev + b1
    h_t = np.tanh(tmp)
    score = W_y @ h_t + b2
    y_hat_t = softmax(score)

    # Save cache for backward
    cache = (y_hat_t, h_t, e_t, h_prev)
    return h_t, y_hat_t, cache


def rnn_step_backward(y_true, dh_t, cache, parameters):
    # Compute local gradients
    y_hat_t, h_t, e_t, h_prev =  cache
    W_e , W_h , b1, W_y, b2 = parameters["W_e"], parameters["W_h"], parameters["b1"], parameters["W_y"], parameters["b2"]
    db2 = dscore = y_hat_t - y_true
    dW_y = dscore @ h_t
    dh_t = W_y.T @ dscore + dh_t # Important !!!
    dtmp = (1 - h_t**2) * dh_t
    dW_e = dtmp @ e_t.T
    dW_h = dtmp @ h_prev.T
    db1 = dtmp
    de_t = W_e.T @ dtmp
    dh_prev = W_h.T @ dtmp
    # Return gradient wrt h_prev and parameters

    parameter_grads = {'W_e': dW_e, 'W_h': dW_h, 'b1': db1, 'b2': db2, 'W_y': dW_y, 'e_t': de_t }
    return dh_prev, parameter_grads